# SiRGraF — Simple Radial Gradient Filter / 단순 동경 기울기 필터 구현

**Paper**: Patel, R., Majumdar, S., Pant, V., Banerjee, D., "A Simple Radial Gradient Filter for Batch-Processing of Coronagraph Images", *Solar Physics* 297, 27 (2022). DOI: 10.1007/s11207-022-01957-y

이 노트북은 SiRGraF 알고리즘의 핵심을 처음부터 구현한다. NumPy 기반의 단순 구현으로, 합성(synthetic) 코로나그래프 영상에 대해 다음을 수행한다:
1. 동경 강도 변화 + 정적 F-corona + 시간 변하는 K-corona (CME) 합성
2. Daily minimum background $I_m$ 계산
3. 방위각 평균으로 uniform background $I_u$ 생성
4. $I' = (I - I_m)/I_u$ 적용 및 NRGF 비교
5. 결과 시각화

This notebook implements the core SiRGraF algorithm from scratch on a synthetic coronagraph dataset and compares it to NRGF.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from typing import Tuple

rng = np.random.default_rng(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## Part 1: Synthetic coronagraph dataset / 합성 코로나그래프 데이터 생성

다음 성분을 합성한다 / We synthesise the following components:
- **F-corona / static background**: $\propto r^{-2.5}$
- **Quiescent K-corona**: $\propto r^{-3}$, 방위각 의존 streamer / azimuth-dependent streamer
- **Dynamic K-corona (CME)**: 일부 frame에서만 추가되는 밝은 호 / a bright arc added in a few frames
- **Photon noise**: Poisson + Gaussian readout

In [ ]:
def make_radial_grid(n: int = 128) -> Tuple[np.ndarray, np.ndarray]:
    """Build a centred radial / azimuthal grid in pixels.

    Args:
        n: Image side length (must be even).

    Returns:
        r: 2-D array of radius (pixels) from image centre.
        phi: 2-D array of azimuth in radians, range [-pi, pi].
    """
    y, x = np.mgrid[:n, :n]
    cx = cy = (n - 1) / 2.0
    r = np.hypot(x - cx, y - cy)
    phi = np.arctan2(y - cy, x - cx)
    return r, phi


def synthesise_frame(r: np.ndarray,
                     phi: np.ndarray,
                     occulter: float = 12.0,
                     fov: float = 60.0,
                     cme_level: float = 0.0,
                     cme_phi0: float = 0.6,
                     cme_dphi: float = 0.4,
                     cme_r0: float = 30.0,
                     cme_dr: float = 8.0,
                     noise: float = 0.02) -> np.ndarray:
    """Synthesise one coronagraph frame.

    Args:
        r: Radius array (pixels).
        phi: Azimuth array (rad).
        occulter: Inner radius of the field (pixels).
        fov: Outer radius of the field (pixels).
        cme_level: Brightness of the synthetic CME (0 disables it).
        cme_phi0: Central azimuth of the CME (rad).
        cme_dphi: Half-width of the CME in azimuth (rad).
        cme_r0: Radial centre of the CME front (pixels).
        cme_dr: Radial half-width of the CME front (pixels).
        noise: Std of additive Gaussian noise on output.

    Returns:
        2-D image with the same shape as `r`.
    """
    # Static F-corona-like component
    f_corona = 5.0 * (np.maximum(r, 1.0) / occulter) ** (-2.5)
    # Quiescent K-corona with two streamers
    k_streamer = 1.5 * (np.maximum(r, 1.0) / occulter) ** (-3.0) * (
        0.4 + 0.6 * np.exp(-((phi - 0.0) / 0.3) ** 2)
        + 0.4 * np.exp(-((phi - 3.0) / 0.3) ** 2))
    img = f_corona + k_streamer
    # Optional CME arc
    if cme_level > 0:
        dphi = np.angle(np.exp(1j * (phi - cme_phi0)))  # signed angular distance
        cme = cme_level * np.exp(-(dphi / cme_dphi) ** 2) * np.exp(-((r - cme_r0) / cme_dr) ** 2)
        img = img + cme
    # Mask occulter and outer FOV
    mask = (r >= occulter) & (r <= fov)
    img = np.where(mask, img, 0.0)
    # Photon-like noise
    img = img + noise * rng.standard_normal(img.shape) * (img > 0)
    return np.clip(img, 0.0, None)


N = 128
r2d, phi2d = make_radial_grid(N)

# Build a 'day' of 30 frames; CME appears in frames 12..18 only.
n_frames = 30
frames = []
for t in range(n_frames):
    has_cme = 12 <= t <= 18
    cme_level = 1.2 if has_cme else 0.0
    cme_r0 = 22.0 + 1.5 * (t - 12)  # CME moves outward
    frames.append(synthesise_frame(r2d, phi2d,
                                   cme_level=cme_level,
                                   cme_r0=cme_r0))
frames = np.stack(frames, axis=0)
print(f'Frames stack shape: {frames.shape}, dtype: {frames.dtype}')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, t in zip(axes, [5, 15, 25]):
    im = ax.imshow(np.log10(frames[t] + 1e-3), cmap='gray', origin='lower')
    ax.set_title(f'Synthetic frame t={t}' + (' (CME)' if 12 <= t <= 18 else ''))
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

## Part 2: Minimum background $I_m$ / 최소 배경

$$I_m(x, y) = \min_{t,\; I_t(x,y) > 0} I_t(x, y)$$

0 픽셀(occulter, 외부 마스크)은 제외 / Exclude zero pixels (occulter, mask).

In [ ]:
def minimum_background(frames: np.ndarray) -> np.ndarray:
    """Per-pixel minimum of positive intensities over a stack of frames.

    Args:
        frames: 3-D array (T, H, W) of Level-1 images.

    Returns:
        2-D minimum-background image (H, W). Pixels that are 0 in every frame
        remain 0.
    """
    masked = np.where(frames > 0, frames, np.inf)
    I_m = masked.min(axis=0)
    I_m = np.where(np.isinf(I_m), 0.0, I_m)
    return I_m


I_m = minimum_background(frames)

fig, ax = plt.subplots(1, 1, figsize=(5, 5))
im = ax.imshow(np.log10(I_m + 1e-3), cmap='gray', origin='lower')
ax.set_title('Minimum background $I_m$')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.04)
plt.show()

## Part 3: Radial profile and uniform background $I_u$ / 동경 프로파일과 균일 배경

$\bar I_m(r) = \langle I_m(r, \phi) \rangle_\phi$, then $I_u(r, \phi) = \bar I_m(r)$ for all $\phi$.

여기서는 polar warp을 명시적으로 하지 않고, 픽셀 반경의 정수 binning으로 동경 프로파일을 추정한다 (구현 단순성 우선).

In [ ]:
def radial_profile(image: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Compute the azimuthally averaged radial profile.

    Args:
        image: 2-D image; zero pixels are treated as 'no data'.
        r: 2-D radius array (pixels) co-registered with `image`.

    Returns:
        1-D array of length max-radius+1, value at each integer radius.
    """
    r_int = r.astype(np.int32)
    rmax = int(r_int.max()) + 1
    weights = (image > 0).astype(np.float64)
    sums = np.bincount(r_int.ravel(), weights=(image * weights).ravel(), minlength=rmax)
    counts = np.bincount(r_int.ravel(), weights=weights.ravel(), minlength=rmax)
    profile = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    return profile


def uniform_background(image: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Build a circularly symmetric background from `image` via radial averaging.

    Args:
        image: 2-D minimum background.
        r: 2-D radius array.

    Returns:
        2-D uniform background with the same shape as `image`.
    """
    profile = radial_profile(image, r)
    return profile[r.astype(np.int32)]


I_u = uniform_background(I_m, r2d)
profile = radial_profile(I_m, r2d)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(profile, lw=2)
axes[0].set_xlabel('radius (pixels)')
axes[0].set_ylabel('mean intensity')
axes[0].set_title('Azimuthally averaged radial profile of $I_m$')
axes[0].grid(alpha=0.3)
im = axes[1].imshow(np.log10(I_u + 1e-3), cmap='gray', origin='lower')
axes[1].set_title('Uniform background $I_u$')
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.04)
plt.tight_layout()
plt.show()

## Part 4: Apply SiRGraF to all frames / SiRGraF 적용

$$I'_t = \frac{I_t - I_m}{I_u}$$

Build the background once, reuse for every frame.

In [ ]:
def sirgraf_batch(frames: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Apply SiRGraF to a stack of frames.

    Args:
        frames: 3-D stack (T, H, W) of Level-1 frames.
        r: 2-D radius array.

    Returns:
        3-D stack of SiRGraF-normalised frames.
    """
    I_m = minimum_background(frames)
    I_u = uniform_background(I_m, r)
    safe = np.where(I_u > 0, I_u, 1.0)
    out = (frames - I_m[None, :, :]) / safe[None, :, :]
    out = np.where((I_u[None, :, :] > 0) & (frames > 0), out, 0.0)
    return out


t0 = time.perf_counter()
filtered = sirgraf_batch(frames, r2d)
t_sirgraf = time.perf_counter() - t0
print(f'SiRGraF batch on {len(frames)} frames: {t_sirgraf*1000:.1f} ms')

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, t in zip(axes, [5, 15, 25]):
    im = ax.imshow(filtered[t], cmap='gray', origin='lower', vmin=-0.05, vmax=0.4)
    ax.set_title(f"SiRGraF $I'_{{t={t}}}$" + (' (CME)' if 12 <= t <= 18 else ''))
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout()
plt.show()

## Part 5: Compare with NRGF / NRGF 비교

$$I'_{\rm NRGF}(r, \phi) = \frac{I(r,\phi) - \mu_\phi(r)}{\sigma_\phi(r)}$$

각 frame마다 평균과 표준편차를 azimuthal하게 추정 — 더 느리고 노이즈에 민감.

Per-frame azimuthal mean and std — slower and noisier.

In [ ]:
def nrgf(frame: np.ndarray, r: np.ndarray) -> np.ndarray:
    """Apply NRGF to a single frame.

    Args:
        frame: 2-D image.
        r: 2-D radius array.

    Returns:
        2-D NRGF-normalised image.
    """
    r_int = r.astype(np.int32)
    rmax = int(r_int.max()) + 1
    valid = (frame > 0).astype(np.float64)
    sums = np.bincount(r_int.ravel(), weights=(frame * valid).ravel(), minlength=rmax)
    sq_sums = np.bincount(r_int.ravel(), weights=((frame ** 2) * valid).ravel(), minlength=rmax)
    counts = np.bincount(r_int.ravel(), weights=valid.ravel(), minlength=rmax)
    mean = np.divide(sums, counts, out=np.zeros_like(sums), where=counts > 0)
    var = np.divide(sq_sums, counts, out=np.zeros_like(sq_sums), where=counts > 0) - mean ** 2
    std = np.sqrt(np.maximum(var, 1e-12))
    mean_2d = mean[r_int]
    std_2d = std[r_int]
    out = np.where((std_2d > 0) & (frame > 0), (frame - mean_2d) / std_2d, 0.0)
    return out


t0 = time.perf_counter()
nrgf_stack = np.stack([nrgf(f, r2d) for f in frames], axis=0)
t_nrgf = time.perf_counter() - t0
print(f'NRGF on {len(frames)} frames: {t_nrgf*1000:.1f} ms (per-frame radial stats)')
print(f'Speed-up SiRGraF / NRGF = {t_nrgf / t_sirgraf:.2f}×')

t = 15
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(np.log10(frames[t] + 1e-3), cmap='gray', origin='lower')
axes[0].set_title(f'Raw frame t={t} (CME)')
axes[0].axis('off')
axes[1].imshow(filtered[t], cmap='gray', origin='lower', vmin=-0.05, vmax=0.4)
axes[1].set_title('SiRGraF')
axes[1].axis('off')
axes[2].imshow(nrgf_stack[t], cmap='gray', origin='lower', vmin=-1.5, vmax=4)
axes[2].set_title('NRGF')
axes[2].axis('off')
plt.tight_layout()
plt.show()

## Part 6: Quantitative contrast metric / 정량 대비 지표

CME mask 영역 안과 밖의 평균 intensity 비를 계산하여 동적 구조 가시성을 정량화한다.

Compute mean intensity inside vs outside a CME mask to quantify dynamic-feature visibility.

In [ ]:
def cme_contrast(filtered_frame: np.ndarray, raw_frame: np.ndarray) -> float:
    """Ratio of mean filtered value inside the brightest 5% of the raw frame to outside.

    Args:
        filtered_frame: enhanced frame.
        raw_frame: original frame for mask construction.

    Returns:
        Contrast ratio (higher = more visible CME).
    """
    valid = raw_frame > 0
    if valid.sum() == 0:
        return 0.0
    thresh = np.quantile(raw_frame[valid], 0.95)
    mask = (raw_frame >= thresh) & valid
    inside = filtered_frame[mask].mean()
    outside = filtered_frame[valid & ~mask].mean()
    return float((inside - outside) / (np.std(filtered_frame[valid]) + 1e-9))


t = 15
c_sir = cme_contrast(filtered[t], frames[t])
c_nrgf = cme_contrast(nrgf_stack[t], frames[t])
c_raw = cme_contrast(frames[t], frames[t])
print(f'Frame t={t} contrast (z-score):  raw={c_raw:.2f}, SiRGraF={c_sir:.2f}, NRGF={c_nrgf:.2f}')

## Summary / 요약

| Concept / 개념 | This paper / 이 논문 | Modern Equivalent / 현대 등가 |
|---|---|---|
| Background separation / 배경 분리 | per-pixel daily minimum | DeForest+ minimum stack, monthly daily-median |
| Radial flattening / 동경 평탄화 | divide by azimuthally averaged $I_m$ | NRGF, FNRGF, MGN, RLMF |
| Batch friendliness / 일괄 처리 | one $O(NHW)$ pass + $O(HW)$/frame | NRGF needs per-frame radial stats |
| Use case / 용도 | CME / dynamic-K-corona visualisation | input normaliser for ML CME segmentation |
| Limitation / 한계 | non-linear → no photometry | quantitative work needs raw + per-frame calibration |

**Key takeaway**: 두 줄의 NumPy 식 ($I_m$ 계산 + $(I-I_m)/I_u$)로 NRGF급 enhancement를 얻을 수 있다. 합성 데이터에서 이미 SiRGraF가 NRGF보다 빠르고 noise 면에서 안정적임을 확인했다.

Two short NumPy expressions reproduce the core SiRGraF algorithm and yield NRGF-quality enhancement on synthetic coronagraph data, with measurable speed and noise-stability advantages.